In [1]:
import os
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput

In [2]:
load_dotenv(override=True)

# Load keys
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")
openrouter_key = os.getenv("OPENROUTER_API_KEY")

# Base URLs
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# 1. Tao async client cho moi provider
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_key)

# 2. Tao model object
llama_model = OpenAIChatCompletionsModel(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    openai_client=groq_client
)

gemini_model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash",
    openai_client=gemini_client
)

qwen_model = OpenAIChatCompletionsModel(
    model="qwen/qwen3.6-plus-preview:free",
    openai_client=openrouter_client
)

In [3]:
# Định nghĩa schemas
from pydantic import BaseModel
from typing import Literal

class PhanTichReview(BaseModel):
    loai: Literal["tich_cuc", "tieu_cuc", "can_xu_ly", "trung_lap"]
    diem_cam_xuc: int
    van_de_chinh: str
    can_boi_thuong: bool

class CauTraLoiReview(BaseModel):
    noi_dung: str
    ton_giong: str
    co_loi_hua: bool
    de_xuat_hanh_dong: str

class KetQuaXuLyReview(BaseModel):
    phan_tich: PhanTichReview
    cau_tra_loi: CauTraLoiReview
    can_escalate: bool


In [4]:
# Guardrail: phát hiện review cần xử lý - không nên để AI tự động trả lời

class KiemTraEscalate(BaseModel):
    can_escalate: bool
    ly_do: str

guardrail_escalate_agent = Agent(
    name="Escalation Checker",
    instructions="""Kiểm tra review có cần chuyển cho người xử lý không.

Cần escalate khi review có:
- Yêu cầu hoàn tiền số lượng lớn (trên 500k)
- Đe dọa kiện tụng hoặc báo chính quyền
- Phản ánh lỗi sản phẩm có thể gây nguy hiểm (điện giật, cháy nổ...)
- Ngôn ngữ thô tục, xúc phạm nặng

Nếu bình thường → can_escalate = False""",
    output_type=KiemTraEscalate,
    model="gpt-4o-mini"
)

@input_guardrail
async def kiem_tra_can_escalate(ctx, agent, message):
    result = await Runner.run(guardrail_escalate_agent, message, context=ctx.context)
    kiem_tra: KiemTraEscalate = result.final_output

    if kiem_tra.can_escalate:
        print(f"Cần escalte: {kiem_tra.ly_do}")

    return GuardrailFunctionOutput(
        output_info={"ly_do_escalate": kiem_tra.ly_do},
        tripwire_triggered=kiem_tra.can_escalate
    )

In [5]:
# Classifier Agent
classifier_agent = Agent(
    name="Review Classifier",
    instructions="""Bạn phân tích đánh giá của khách hàng Việt Nam trên Shopee/TikTok Shop.

Phân loại chính xác:
- tich_cuc: khách hài lòng, khen ngợi
- tieu_cuc: khách không hài lòng, phàn nàn về sản phẩm/ship/dịch vụ
- can_xu_ly: có vấn đề cụ thể cần giải quyết (thiếu hàng, hàng lỗi...)
- trung_lap: không rõ cảm xúc, chỉ nhận xét khách quan

Đánh giá điểm cảm xúc 1-10 và tóm tắt vấn đề chính.""",
    output_type=PhanTichReview,
    model="gpt-4o-mini",
    input_guardrails=[kiem_tra_can_escalate]  # ← chuyển guardrail vào đây
)

In [6]:
# Response Agents with multi-model

# Review tiêu cực -> llama
agent_xu_ly_tieu_cuc = Agent(
    name="Xử lý Review Tiêu Cực", 
    instructions="""Bạn đại diện shop trả lời review tiêu cực của khách hàng Việt Nam.

Nguyên tắc:
- Xin lỗi chân thành, không đổ lỗi cho khách
- Thể hiện đồng cảm với bất tiện của khách
- Đưa ra hướng giải quyết cụ thể (inbox shop, đổi hàng...)
- KHÔNG cam kết hoàn tiền nếu chưa xem xét cụ thể
- Tông: chân thành, kiên nhẫn

Giữ dưới 80 từ, tiếng Việt tự nhiên.""",
    output_type=CauTraLoiReview,
    model=llama_model
)

# Review tích cực -> Gemini
agent_xu_ly_tich_cuc = Agent(
    name="Xử lý Review Tích Cực",
    instructions="""Bạn đại diện shop trả lời review tích cực của khách hàng.

Phong cách: vui vẻ, cảm ơn chân thành, khuyến khích mua lại.
Có thể mention chương trình loyalty/voucher nếu phù hợp.
Dưới 50 từ, tự nhiên không sáo rỗng.""",
    output_type=CauTraLoiReview,
    model=gemini_model
)

# Review trung lập/ cần xử lý -> GPT-4o-mini
agent_xu_ly_trung_lap = Agent(
    name="Xử lý Review Trung Lập",
    instructions="""Bạn đại diện shop trả lời review trung lập hoặc có vấn đề cần xử lý.

Phong cách: chuyên nghiệp, hữu ích, rõ ràng.
Cảm ơn feedback, giải thích hoặc đề nghị hỗ trợ thêm nếu cần.
Dưới 60 từ.""",
    output_type=CauTraLoiReview,
    model="gpt-4o-mini"
)

In [7]:
# ReviewBot orchestrator
async def xu_ly_review(noi_dung_review: str) -> KetQuaXuLyReview | None:

    print(f"\n Review: {noi_dung_review[:80]}..")

    # Tao orchestrator agent voi guardrail
    review_bot = Agent(
        name="ReviewBot Orchestrator",
        instructions="Phân tích review và điều phối xử lý",
        model="gpt-4o-mini",
        input_guardrails=[kiem_tra_can_escalate]
    )

    # 1. Kiem tra escalate (guardrail)
    try:
        with trace(f"reviewbot"):
            # Phan loai review
            phan_tich_result = await Runner.run(
                classifier_agent, noi_dung_review
            )
            phan_tich: PhanTichReview = phan_tich_result.final_output
            print(f" Phân loại: {phan_tich.loai} (điểm: {phan_tich.diem_cam_xuc}/10)")

            if phan_tich.loai == "tich_cuc":
                agent_xu_ly = agent_xu_ly_tich_cuc
            elif phan_tich.loai in ["tieu_cuc", "can_xu_ly"]:
                agent_xu_ly = agent_xu_ly_tieu_cuc
            else:
                agent_xu_ly = agent_xu_ly_trung_lap

            tra_loi_result = await Runner.run(
                agent_xu_ly,
                f"Reveiw: {noi_dung_review}\nVấn đề chính: {phan_tich.van_de_chinh}"
            )
            cau_tra_loi: CauTraLoiReview = tra_loi_result.final_output
            return KetQuaXuLyReview(
                phan_tich=phan_tich,
                cau_tra_loi=cau_tra_loi,
                can_escalate=False
            )
    except Exception as e:
        if "tripwire" in str(e).lower() or "guardrail" in str(e).lower():
            print(f"Review cần xử lý thủ công - chuyển cho nhân viên")
            return None
        raise e


In [8]:
# test review thuc te
async def demo_reviewbot():
    reviews = [
        # Review tích cực
        "Hàng đẹp lắm ạ, đúng như mô tả. Ship nhanh, đóng gói cẩn thận. "
        "Sẽ ủng hộ shop dài dài! ⭐⭐⭐⭐⭐",

        # Review tiêu cực — có thể xử lý tự động
        "Hàng nhận được bị bể góc, không đúng màu như ảnh. "
        "Thất vọng quá, mình đặt làm quà tặng mà giờ xấu hổ ghê.",

        # Review cần escalate — điện bị lỗi nguy hiểm
        "Sạc điện thoại này dùng 2 ngày thì phát nổ, may mà không ai bị thương. "
        "Tôi sẽ kiện lên Bộ Công thương và đăng lên báo nếu không được giải quyết.",

        # Review trung lập
        "Sản phẩm tạm ổn, không tệ nhưng cũng không xuất sắc. "
        "Giá hơi cao so với chất lượng.",
    ]

    for review in reviews:
        ket_qua = await xu_ly_review(review)

        if ket_qua is None:
            print(" -> Chuyển qua cho nhân viên xử lý")
        else:
            print(f"Câu trả lời ({ket_qua.cau_tra_loi.ton_giong}):")
            print(ket_qua.cau_tra_loi.noi_dung)
            if ket_qua.cau_tra_loi.de_xuat_hanh_dong:
                print(f"Nhân viên cần: {ket_qua.cau_tra_loi.de_xuat_hanh_dong}")

await demo_reviewbot()



 Review: Hàng đẹp lắm ạ, đúng như mô tả. Ship nhanh, đóng gói cẩn thận. Sẽ ủng hộ shop dà..
 Phân loại: tich_cuc (điểm: 10/10)
Câu trả lời (Vui vẻ, biết ơn, thân thiện):
Cảm ơn bạn yêu rất nhiều vì review 5 sao cực xịn ạ! Shop vui lắm khi bạn ưng ý từ sản phẩm đến dịch vụ. Đừng quên ghé lại shop thường xuyên để nhận thêm nhiều ưu đãi độc quyền dành cho khách thân thiết nhé!
Nhân viên cần: Khách hàng quay lại mua sắm và nhận ưu đãi

 Review: Hàng nhận được bị bể góc, không đúng màu như ảnh. Thất vọng quá, mình đặt làm qu..
 Phân loại: tieu_cuc (điểm: 3/10)
Câu trả lời (chân thành, kiên nhẫn):
Xin lỗi vì hàng nhận được không đạt yêu cầu của bạn. Chúng tôi hiểu sự thất vọng khi nhận hàng bị bể góc và không đúng màu như ảnh, đặc biệt khi dùng làm quà tặng.
Nhân viên cần: Bạn vui lòng inbox cho shop để được hỗ trợ đổi hàng hoặc xử lý vấn đề này nhé

 Review: Sạc điện thoại này dùng 2 ngày thì phát nổ, may mà không ai bị thương. Tôi sẽ ki..
Cần escalte: Rèn sạc điện thoại bị phấn nổ,